# Claims Reserve Forecasting

## Problem Statement

### Business Context

Insurance carriers must set **case reserves** — estimates of future claim payouts — accurately and early. Under-reserving creates solvency risk; over-reserving ties up capital and distorts pricing.

You are a data scientist at a workers' compensation carrier. Leadership wants a **regression model** that forecasts **total reserve** at claim intake using demographic, clinical, provider, and historical claimant features — so actuaries can triage high-exposure claims and improve reserve adequacy.

### Data Description

**Target**
* **total_reserve** — 6-month case reserve (`reserve_6` in source data)

**Features** (engineered from `claims_data.csv`)
* **patient_age** — claimant age at date of event
* **diagnosis_code** — injury/diagnosis category (`our_cause_3`)
* **procedure_code** — body part / procedure category (`our_cause_4`)
* **admission_type** — claim admission type (`claim_type`: MO, OT, IO)
* **days_in_hospital** — days from claim open to close (when available)
* **provider_type** — processing unit (`proc_unit`)
* **injury_severity** — injury mechanism (`our_cause_2`)
* **num_previous_claims** — prior claims for same claimant (SSN)
* **avg_previous_reserve** — average reserve on prior claims for claimant
* **initial_estimate** — initial payment estimate (`amount`)
* **reported_delay_days** — days to CMS reporting (`days_to_cms`)
* **state** — claimant state (`clmnt_state`)

##### What is Total Reserve?

Total reserve is the carrier's estimated remaining liability on a claim — the amount expected to be paid beyond what has already been paid.

##### Relation between Initial Estimate and Reserve

Initial estimate reflects early exposure signals; total reserve incorporates expected future development. Claims with higher severity, longer hospital stays, and prior high-reserve history tend to carry higher reserves.

### **Please read the instructions carefully before starting the project.**
This is a commented Jupyter IPython Notebook file in which all the instructions and tasks to be performed are mentioned.
* Run cells **sequentially** from top to bottom.
* Feature engineering maps raw `claims_data.csv` columns to modeling features listed above.
* The best tuned model is serialized to `models/best_reserve_model.pkl` for FastAPI deployment on Azure.
* Add results/observations derived from the analysis to your presentation.

## Importing necessary libraries

In [ ]:
# Installing the libraries with the specified version.
# uncomment and run the following line if Google Colab is being used
# !pip install scikit-learn==1.2.2 seaborn==0.13.1 matplotlib==3.7.1 numpy==1.25.2 pandas==1.5.3 imbalanced-learn==0.10.1 xgboost==2.0.3 joblib==1.3.2 -q --user

In [ ]:
# To load and manipulate data
import pandas as pd
import numpy as np
import os
import joblib

# To visualize data
import matplotlib.pyplot as plt
import seaborn as sns

# To be used for missing value imputation
from sklearn.impute import SimpleImputer

# Import evaluation metrics for regression
from sklearn import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

%matplotlib inline

# To ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Import SMOTE package for oversampling
from imblearn.over_sampling import SMOTE

# Import Random undersampler for undersampling
from imblearn.under_sampling import RandomUnderSampler

# Import packages to split data
from sklearn.model_selection import train_test_split

# Import package for RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Import package GridSearchCV
from sklearn.model_selection import GridSearchCV

# Import packages for regression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
)
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

RS = 1

In [ ]:
# Installing the libraries with the specified version.
# uncomment and run the following lines if Jupyter Notebook is being used
# !pip install scikit-learn==1.2.2 seaborn==0.13.1 matplotlib==3.7.1 numpy==1.25.2 pandas==1.5.3 imbalanced-learn==0.10.1 xgboost==2.0.3 joblib==1.3.2 -q --user

## Loading the dataset

In [ ]:
# Mount google drive (Colab only)
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Read .csv file in dataframe
orig_df = pd.read_csv('data/claims_data.csv')

In [ ]:
# creating a copy of the data
claims_df = orig_df.copy()

## Feature Engineering

In [ ]:
# Map raw claims columns to modeling features
claims_df['date_event_dt'] = pd.to_datetime(claims_df['date_event'], format='%m/%d/%Y', errors='coerce')
claims_df['clmnt_dob_dt'] = pd.to_datetime(claims_df['clmnt_dob'], format='%m/%d/%Y', errors='coerce')
claims_df['date_open_dt'] = pd.to_datetime(claims_df['date_open'], format='%m/%d/%Y', errors='coerce')
claims_df['date_close_dt'] = pd.to_datetime(claims_df['date_close'], format='%m/%d/%Y', errors='coerce')

claims_df['patient_age'] = ((claims_df['date_event_dt'] - claims_df['clmnt_dob_dt']).dt.days / 365.25).round(1)
claims_df['days_in_hospital'] = (claims_df['date_close_dt'] - claims_df['date_open_dt']).dt.days

claims_df['diagnosis_code'] = claims_df['our_cause_3']
claims_df['procedure_code'] = claims_df['our_cause_4']
claims_df['admission_type'] = claims_df['claim_type']
claims_df['provider_type'] = claims_df['proc_unit'].astype(str)
claims_df['injury_severity'] = claims_df['our_cause_2']
claims_df['initial_estimate'] = claims_df['amount']
claims_df['reported_delay_days'] = claims_df['days_to_cms']
claims_df['state'] = claims_df['clmnt_state']
claims_df['total_reserve'] = claims_df['reserve_6']

# Sort for claimant history features
claims_df = claims_df.sort_values(['clmnt_ssn', 'date_open_dt']).reset_index(drop=True)
claims_df['num_previous_claims'] = claims_df.groupby('clmnt_ssn').cumcount()
claims_df['avg_previous_reserve'] = (
    claims_df.groupby('clmnt_ssn')['total_reserve']
    .apply(lambda s: s.shift(1).expanding().mean())
    .reset_index(level=0, drop=True)
    .fillna(0)
)

MODEL_FEATURES = [
    'patient_age', 'diagnosis_code', 'procedure_code', 'admission_type',
    'days_in_hospital', 'provider_type', 'injury_severity',
    'num_previous_claims', 'avg_previous_reserve', 'initial_estimate',
    'reported_delay_days', 'state'
]
TARGET = 'total_reserve'

model_df = claims_df[MODEL_FEATURES + [TARGET, 'claim_uid']].copy()
model_df.head()

## Data Overview

### Viewing the first, last and random 5 rows of the dataset

In [ ]:
# Print the head of data set
model_df.head()

In [ ]:
# Print the Tail of data set
model_df.tail()

In [ ]:
# Print Random rows of data set
model_df.sample(5, random_state=RS)

- Observations
- Sanity checks on engineered features and target distribution.

### Checking the shape of the dataset.

In [ ]:
# Print the rows and columns of data set
print("Number of Rows: ", model_df.shape[0])
print("Number of Columns: ", model_df.shape[1])

### Checking the attribute types

In [ ]:
# Print the dataset info
model_df.info()

### Checking the statistical summary

In [ ]:
# Print the statistical summary including categorical variables
model_df.describe(include='all').T

### Checking for missing values

In [ ]:
# Check if data set has any missing values
model_df.isnull().sum().sort_values(ascending=False)

### Checking for duplicate values

In [ ]:
# Check if dataset has duplicate rows
print(f"Duplicate rows: {model_df.duplicated().sum()}")

In [ ]:
# Drop the claim_uid column from the data set as it is not required in further EDA and Data Modeling
model_df.drop('claim_uid', axis=1, inplace=True, errors='ignore')

## Exploratory Data Analysis (EDA)

#### The below functions need to be defined to carry out the Exploratory Data Analysis.

In [ ]:
# function to plot a boxplot and a histogram along the same scale.


def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    # Boxplot and histogram combined
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,
        sharex=True,
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )
    ax_hist2.axvline(data[feature].mean(), color="green", linestyle="--")
    ax_hist2.axvline(data[feature].median(), color="black", linestyle="-")

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    # Barplot with percentage at the top

    total = len(data[feature])
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(max(count + 1, 8), 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n].sort_values() if n else data[feature].value_counts().index,
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(100 * p.get_height() / total)
        else:
            label = p.get_height()

        x = p.get_x() + p.get_width() / 2
        y = p.get_height()

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )

    plt.show()

In [ ]:
# function to plot stacked bar chart

def stacked_barplot(data, predictor, target):
    # Print category counts and plot a stacked bar chart
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(max(count + 1, 8), 5))
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
### Function to plot distributions (adapted for regression target)


def _bin_reserve_target(data, target):
    data = data.copy()
    threshold = data[target].median()
    data['_reserve_bin'] = np.where(
        data[target] > threshold, 'High Reserve', 'Low Reserve'
    )
    return data


def distribution_plot_wrt_target(data, predictor, target):
    # Compare predictor distribution for high vs low reserve claims

    binned = _bin_reserve_target(data, target)
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = binned['_reserve_bin'].unique()

    axs[0, 0].set_title("Distribution for " + str(target_uniq[0]))
    sns.histplot(
        data=binned[binned['_reserve_bin'] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
    )

    axs[0, 1].set_title("Distribution for " + str(target_uniq[1]))
    sns.histplot(
        data=binned[binned['_reserve_bin'] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
    )

    axs[1, 0].set_title("Boxplot w.r.t reserve level")
    sns.boxplot(data=binned, x='_reserve_bin', y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t reserve level")
    sns.boxplot(
        data=binned,
        x='_reserve_bin',
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

### Univariate Analysis

#### **Let's look at spread of the Age of the Claimants**

In [ ]:
# Create the Histogram and boxplot for patient_age
histogram_boxplot(model_df, 'patient_age')

**Observations**
- Distribution shape and central tendency for reserve drivers.

#### **Let's look at spread of the Days in Hospital**

In [ ]:
# Create the Histogram and boxplot for days_in_hospital
histogram_boxplot(model_df, 'days_in_hospital')

**Observations**
- Distribution shape and central tendency for reserve drivers.

#### **Let's look at spread of the Number of Previous Claims**

In [ ]:
# Create the Histogram and boxplot for num_previous_claims
histogram_boxplot(model_df, 'num_previous_claims')

**Observations**
- Distribution shape and central tendency for reserve drivers.

#### **Let's look at spread of the Average Previous Reserve**

In [ ]:
# Create the Histogram and boxplot for avg_previous_reserve
histogram_boxplot(model_df, 'avg_previous_reserve')

**Observations**
- Distribution shape and central tendency for reserve drivers.

#### **Let's look at spread of the Initial Estimate**

In [ ]:
# Create the Histogram and boxplot for initial_estimate
histogram_boxplot(model_df, 'initial_estimate')

**Observations**
- Distribution shape and central tendency for reserve drivers.

#### **Let's look at spread of the Reported Delay Days**

In [ ]:
# Create the Histogram and boxplot for reported_delay_days
histogram_boxplot(model_df, 'reported_delay_days')

**Observations**
- Distribution shape and central tendency for reserve drivers.

#### **Let's look at spread of the Total Reserve (Target)**

In [ ]:
# Create the Histogram and boxplot for total_reserve
histogram_boxplot(model_df, 'total_reserve')

**Observations**
- Distribution shape and central tendency for reserve drivers.

#### **Let's look at Diagnosis Code counts**

In [ ]:
# Check unique values in diagnosis_code
print(f"Total unique diagnosis_code = {model_df['diagnosis_code'].nunique()} with values {model_df['diagnosis_code'].dropna().unique()[:10]}")

# Percentage per category
labeled_barplot(model_df, 'diagnosis_code', perc=True)

**Observations**
- Category concentration and potential cardinality issues.

#### **Let's look at Procedure Code counts**

In [ ]:
# Check unique values in procedure_code
print(f"Total unique procedure_code = {model_df['procedure_code'].nunique()} with values {model_df['procedure_code'].dropna().unique()[:10]}")

# Percentage per category
labeled_barplot(model_df, 'procedure_code', perc=True)

**Observations**
- Category concentration and potential cardinality issues.

#### **Let's look at Admission Type counts**

In [ ]:
# Check unique values in admission_type
print(f"Total unique admission_type = {model_df['admission_type'].nunique()} with values {model_df['admission_type'].dropna().unique()[:10]}")

# Percentage per category
labeled_barplot(model_df, 'admission_type', perc=True)

**Observations**
- Category concentration and potential cardinality issues.

#### **Let's look at Provider Type counts**

In [ ]:
# Check unique values in provider_type
print(f"Total unique provider_type = {model_df['provider_type'].nunique()}")

# Percentage per category (top 15)
labeled_barplot(model_df, 'provider_type', perc=True, n=15)

**Observations**
- Category concentration and potential cardinality issues.

#### **Let's look at Injury Severity counts**

In [ ]:
# Check unique values in injury_severity
print(f"Total unique injury_severity = {model_df['injury_severity'].nunique()} with values {model_df['injury_severity'].dropna().unique()[:10]}")

# Percentage per category
labeled_barplot(model_df, 'injury_severity', perc=True)

**Observations**
- Category concentration and potential cardinality issues.

#### **Let's look at State counts**

In [ ]:
# Check unique values in state
print(f"Total unique state = {model_df['state'].nunique()}")

# Percentage per category (top 15)
labeled_barplot(model_df, 'state', perc=True, n=15)

**Observations**
- Category concentration and potential cardinality issues.

### Multivariate Analysis

#### **Let's look at the relationship between Total Reserve and Patient Age**

In [ ]:
# Relationship between total_reserve and patient_age
distribution_plot_wrt_target(model_df, 'patient_age', 'total_reserve')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Diagnosis Code**

In [ ]:
# Relationship between total_reserve and diagnosis_code
binned = _bin_reserve_target(model_df, 'total_reserve')
stacked_barplot(binned, 'diagnosis_code', '_reserve_bin')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Procedure Code**

In [ ]:
# Relationship between total_reserve and procedure_code
binned = _bin_reserve_target(model_df, 'total_reserve')
stacked_barplot(binned, 'procedure_code', '_reserve_bin')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Admission Type**

In [ ]:
# Relationship between total_reserve and admission_type
binned = _bin_reserve_target(model_df, 'total_reserve')
stacked_barplot(binned, 'admission_type', '_reserve_bin')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Days In Hospital**

In [ ]:
# Relationship between total_reserve and days_in_hospital
distribution_plot_wrt_target(model_df, 'days_in_hospital', 'total_reserve')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Provider Type**

In [ ]:
# Relationship between total_reserve and provider_type
binned = _bin_reserve_target(model_df, 'total_reserve')
stacked_barplot(binned, 'provider_type', '_reserve_bin')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Injury Severity**

In [ ]:
# Relationship between total_reserve and injury_severity
binned = _bin_reserve_target(model_df, 'total_reserve')
stacked_barplot(binned, 'injury_severity', '_reserve_bin')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Num Previous Claims**

In [ ]:
# Relationship between total_reserve and num_previous_claims
distribution_plot_wrt_target(model_df, 'num_previous_claims', 'total_reserve')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Avg Previous Reserve**

In [ ]:
# Relationship between total_reserve and avg_previous_reserve
distribution_plot_wrt_target(model_df, 'avg_previous_reserve', 'total_reserve')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Initial Estimate**

In [ ]:
# Relationship between total_reserve and initial_estimate
distribution_plot_wrt_target(model_df, 'initial_estimate', 'total_reserve')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and Reported Delay Days**

In [ ]:
# Relationship between total_reserve and reported_delay_days
distribution_plot_wrt_target(model_df, 'reported_delay_days', 'total_reserve')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the relationship between Total Reserve and State**

In [ ]:
# Relationship between total_reserve and state
binned = _bin_reserve_target(model_df, 'total_reserve')
stacked_barplot(binned, 'state', '_reserve_bin')

**Observations**
- Feature behavior across high vs low reserve claims.

#### **Let's look at the co-relation between all numeric columns**

In [ ]:
# Select numeric columns for correlation
numeric_cols = model_df.select_dtypes(include=[np.number]).columns.tolist()
plt.figure(figsize=(14, 10))
sns.heatmap(model_df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()

**Observations**

**Multicollinearity:**
- **initial_estimate** and payment-related fields may correlate with reserve outcomes.
- **avg_previous_reserve** and **num_previous_claims** capture claimant history.

**Reserve Drivers:**
- Non-zero reserves are concentrated in claims with specific injury types and longer reporting delays.

In [ ]:
# Pairplot for key numeric features
pair_cols = ['patient_age', 'initial_estimate', 'avg_previous_reserve', 'total_reserve']
sns.pairplot(data=model_df[pair_cols].dropna(), corner=True, height=1.5);

## Data Pre-processing


### Outlier detection and treatment

In [ ]:
# Get the 25th percentile for each numeric column
Quarter1 = model_df.select_dtypes(include=["float64", "int64"]).quantile(0.25)

# Get the 75th percentile for each numeric column
Quarter3 = model_df.select_dtypes(include=["float64", "int64"]).quantile(0.75)

# Calculate Inter Quantile Range (75th percentile - 25th percentile)
IQR = Quarter3 - Quarter1

# Find lower and upper bounds for all values. All values outside these bounds are outliers
lower = Quarter1 - 1.5 * IQR
upper = Quarter3 + 1.5 * IQR

print("Lower bounds (sample):")
print(lower.head(10))
print("\nUpper bounds (sample):")
print(upper.head(10))

In [ ]:
# Outliers are retained for tree-based models; reserve amounts are naturally right-skewed.
# Cap extreme total_reserve at 99th percentile to reduce leverage of extreme cases
cap_value = model_df['total_reserve'].quantile(0.99)
model_df['total_reserve'] = model_df['total_reserve'].clip(upper=cap_value)
print(f"Capped total_reserve at 99th percentile: {cap_value:.2f}")

### Train-Val-Test Split

In [ ]:
# defining the explanatory (independent) and response (dependent) variables
X = model_df.drop([TARGET], axis=1)
y = model_df[TARGET]

In [ ]:
# splitting the data in an 80:20 ratio for train and Test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RS)

# splitting the data in a 75:25 ratio for train and Validation sets
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RS)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

### Missing value imputation


In [ ]:
# Instanciate SimpleImputer function to impute the values as a Mode in categorical columns
imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')

# Prepare list of categorical columns needing missing value treatment
cat_cols = ['diagnosis_code', 'procedure_code', 'injury_severity']

# fit and transform the imputer on train data
X_train[cat_cols] = imputer.fit_transform(X_train[cat_cols])
X_val[cat_cols] = imputer.transform(X_val[cat_cols])
X_test[cat_cols] = imputer.transform(X_test[cat_cols])

# Median imputation for numeric columns with missing values
num_imputer = SimpleImputer(missing_values=np.nan, strategy='median')
num_cols = ['days_in_hospital', 'reported_delay_days', 'patient_age']

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_val[num_cols] = num_imputer.transform(X_val[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

### Encoding Categorial Variables

In [ ]:
# Create dummies for all categorical columns in Train, Validation and Test data set
X_train = pd.get_dummies(X_train, drop_first=True)
X_val = pd.get_dummies(X_val, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns across splits
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Encoded feature count: {X_train.shape[1]}")

## Model Building

### Model evaluation criterion

For reserve forecasting we optimize **RMSE** (primary) while tracking **MAE**, **R²**, and **MAPE**.

**Business metric:** **% predictions within 10% of actual** — the share of claims where `|predicted - actual| / actual <= 0.10` (excluding zero-actual rows).

**Which metric to optimize?**
* RMSE penalizes large reserve errors heavily — critical for tail-risk claims.
* MAPE supports business interpretability.
* Within-10% rate reflects operational adequacy for actuarial triage.

In [ ]:
# defining a function to compute regression metrics
def model_performance_regression_sklearn(model, predictors, target):
    # Compute regression metrics and business KPI
    pred = model.predict(predictors)
    pred = np.maximum(pred, 0)  # reserves cannot be negative

    rmse = np.sqrt(mean_squared_error(target, pred))
    mae = mean_absolute_error(target, pred)
    r2 = r2_score(target, pred)

    # MAPE — exclude zero actuals to avoid division by zero
    mask = np.array(target) != 0
    if mask.sum() > 0:
        mape = np.mean(np.abs((np.array(target)[mask] - pred[mask]) / np.array(target)[mask])) * 100
    else:
        mape = np.nan

    # Business metric: % within 10% of actual
    if mask.sum() > 0:
        within_10 = np.mean(np.abs((np.array(target)[mask] - pred[mask]) / np.array(target)[mask]) <= 0.10) * 100
    else:
        within_10 = np.nan

    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2,
            "MAPE": mape,
            "Pct_Within_10pct": within_10,
        },
        index=[0],
    )
    return df_perf, pred


def plot_actual_vs_predicted(model, predictors, target, title="Actual vs Predicted"):
    _, pred = model_performance_regression_sklearn(model, predictors, target)
    plt.figure(figsize=(8, 6))
    plt.scatter(target, pred, alpha=0.5, color="teal")
    max_val = max(np.max(target), np.max(pred))
    plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')
    plt.xlabel('Actual Total Reserve')
    plt.ylabel('Predicted Total Reserve')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_residuals(model, predictors, target, title="Residual Plot"):
    _, pred = model_performance_regression_sklearn(model, predictors, target)
    residuals = np.array(target) - pred
    plt.figure(figsize=(8, 6))
    plt.scatter(pred, residuals, alpha=0.5, color="orange")
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel('Predicted Total Reserve')
    plt.ylabel('Residual (Actual - Predicted)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Helper: SMOTE oversampling for regression (balance high vs zero reserve, KNN target for synthetics)
def smote_oversample_regression(X, y, random_state=1):
    y_bin = (y > 0).astype(int)
    k = min(5, max(1, (y_bin == 1).sum() - 1))
    sm = SMOTE(sampling_strategy=1, k_neighbors=k, random_state=random_state)
    X_res, _ = sm.fit_resample(X, y_bin)
    X_res = pd.DataFrame(X_res, columns=X.columns)
    knn = KNeighborsRegressor(n_neighbors=min(5, len(X)))
    knn.fit(X, y)
    y_res = knn.predict(X_res)
    y_res = np.maximum(y_res, 0)
    return X_res, pd.Series(y_res, name=y.name)


# Helper: undersample majority class (zero reserve) preserving continuous target
def undersample_regression(X, y, random_state=1):
    y_bin = (y > 0).astype(int)
    rus = RandomUnderSampler(random_state=random_state, sampling_strategy=1)
    idx = np.arange(len(X)).reshape(-1, 1)
    idx_res, _ = rus.fit_resample(idx, y_bin)
    indices = idx_res.flatten()
    return X.iloc[indices].reset_index(drop=True), y.iloc[indices].reset_index(drop=True)

### Model Building with original data

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Bagging", BaggingRegressor(random_state=RS)))
models.append(("Random forest", RandomForestRegressor(random_state=RS)))
models.append(("GBM", GradientBoostingRegressor(random_state=RS)))
models.append(("Adaboost", AdaBoostRegressor(random_state=RS)))
models.append(("Xgboost", XGBRegressor(random_state=RS, objective='reg:squarederror')))
models.append(("dtree", DecisionTreeRegressor(random_state=RS)))

In [ ]:
models_train_score = []
models_validation_score = []

for name, model in models:
    model.fit(X_train, y_train)
    pred_train = np.maximum(model.predict(X_train), 0)
    pred_val = np.maximum(model.predict(X_val), 0)
    scores = np.sqrt(mean_squared_error(y_train, pred_train))
    scores_val = np.sqrt(mean_squared_error(y_val, pred_val))
    models_train_score.append((name, scores))
    models_validation_score.append((name, scores_val))

models_train_score_df = pd.DataFrame(models_train_score, columns=['Model Name', 'Score'])
models_validation_score_df = pd.DataFrame(models_validation_score, columns=['Model Name', 'Score'])

models_score_df = pd.merge(
    models_train_score_df,
    models_validation_score_df,
    on='Model Name',
    suffixes=('_train_original', '_val_original')
)
print("RMSE comparison — original data")
models_score_df

### Model Building with Oversampled data


In [ ]:
# Original label counts (zero vs non-zero reserve)
print("Original Count of Labels \n")
print(f"Total counts of Non-Zero Reserve : {sum(y_train > 0)}")
print(f"Total counts of Zero Reserve: {sum(y_train == 0)} \n")

In [ ]:
# Create synthetic data to treat imbalanced reserve distribution
X_train_over, y_train_over = smote_oversample_regression(X_train, y_train, random_state=RS)

In [ ]:
# Print the Oversampled Count of Labels
print("Oversampled Count of Labels \n")
print(f"Total counts of Non-Zero Reserve : {sum(y_train_over > 0)}")
print(f"Total counts of Zero Reserve: {sum(y_train_over == 0)} \n")

print("After Oversampling\n")
print(f"The shape of train_X: {X_train_over.shape}")
print(f"The shape of train_y: {y_train_over.shape} \n")

In [ ]:
models_train_score_over = []
models_validation_score_over = []

for name, model in models:
    model.fit(X_train_over, y_train_over)
    pred_train = np.maximum(model.predict(X_train_over), 0)
    pred_val = np.maximum(model.predict(X_val), 0)
    scores = np.sqrt(mean_squared_error(y_train_over, pred_train))
    scores_val = np.sqrt(mean_squared_error(y_val, pred_val))
    models_train_score_over.append((name, scores))
    models_validation_score_over.append((name, scores_val))

models_train_score_over_df = pd.DataFrame(models_train_score_over, columns=['Model Name', 'Score'])
models_validation_score_over_df = pd.DataFrame(models_validation_score_over, columns=['Model Name', 'Score'])

models_score_over_df = pd.merge(
    models_train_score_over_df,
    models_validation_score_over_df,
    on='Model Name',
    suffixes=('_train_oversampled', '_val_oversampled')
)
print("RMSE comparison — oversampled data")
models_score_over_df

### Model Building with Undersampled data

In [ ]:
print("Original Count of Labels \n")
print(f"Total counts of Non-Zero Reserve : {sum(y_train > 0)}")
print(f"Total counts of Zero Reserve: {sum(y_train == 0)} \n")

In [ ]:
# Random undersampler for under sampling the data
X_train_un, y_train_un = undersample_regression(X_train, y_train, random_state=RS)

In [ ]:
print("Undersampled Count of Labels \n")
print(f"Total counts of Non-Zero Reserve: {sum(y_train_un > 0)}")
print(f"Total counts of Zero Reserve: {sum(y_train_un == 0)} \n")

print("After Undersampling\n")
print(f"The shape of train_X: {X_train_un.shape}")
print(f"The shape of train_y: {y_train_un.shape} \n")

In [ ]:
models_train_score_under = []
models_validation_score_under = []

for name, model in models:
    model.fit(X_train_un, y_train_un)
    pred_train = np.maximum(model.predict(X_train_un), 0)
    pred_val = np.maximum(model.predict(X_val), 0)
    scores = np.sqrt(mean_squared_error(y_train_un, pred_train))
    scores_val = np.sqrt(mean_squared_error(y_val, pred_val))
    models_train_score_under.append((name, scores))
    models_validation_score_under.append((name, scores_val))

models_train_score_under_df = pd.DataFrame(models_train_score_under, columns=['Model Name', 'Score'])
models_validation_score_under_df = pd.DataFrame(models_validation_score_under, columns=['Model Name', 'Score'])

models_score_under_df = pd.merge(
    models_train_score_under_df,
    models_validation_score_under_df,
    on='Model Name',
    suffixes=('_train_undersampled', '_val_undersampled')
)
print("RMSE comparison — undersampled data")
models_score_under_df

### Score comparison on Training and Validation score for basic models

In [ ]:
# merge models_score_df, models_score_over_df and models_score_under_df into one data frame
print("RMSE Score comparison on Training and Validation score for basic models\n")
models_score_final_df = pd.merge(models_score_df, models_score_over_df, on='Model Name')
models_score_final_df = pd.merge(models_score_final_df, models_score_under_df, on='Model Name')
models_score_final_df

## HyperparameterTuning

#### Sample Parameter Grids

**Note**

1. Sample parameter grids are provided for hyperparameter tuning. These grids balance performance improvement and execution time.
2. Scoring metric: **neg_root_mean_squared_error** (maximize = lower RMSE).

- For Gradient Boosting:

```
param_grid = {
    "n_estimators": np.arange(50, 110, 25),
    "learning_rate": [0.01, 0.1, 0.05],
    "subsample": [0.7, 0.9],
    "max_features": [0.5, 0.7, 1],
}
```

- For Adaboost, Bagging, Random Forest, Decision Trees, XGBoost — see tuning cells below.

#### Tuned Decision tree

In [ ]:
# defining model
dt_tuned_model = DecisionTreeRegressor(random_state=RS)

# Parameter grid to pass in RandomSearchCV
param_grid = {'max_depth': np.arange(2,6),
              'min_samples_leaf': [1, 4, 7],
              'max_leaf_nodes' : [10,15],
              'min_impurity_decrease': [0.0001,0.001] }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=True)

# Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=dt_tuned_model, param_distributions=param_grid, n_iter=10, n_jobs=-1, scoring=scorer, cv=5, random_state=RS)

# Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(randomized_cv.best_params_, randomized_cv.best_score_))

##### Tuned Decision tree Model with Original Data

In [ ]:
# Build tuned model with best parameters from RandomizedSearchCV
dt_tuned_model = randomized_cv.best_estimator_

# Train model with original data set
dt_tuned_model.fit(X_train, y_train)

In [ ]:
# Calculating different metrics on train set
dt_train = model_performance_regression_sklearn(dt_tuned_model, X_train, y_train)[0]
print("Training performance:")
dt_train

In [ ]:
# Calculating different metrics on Validation set
dt_val = model_performance_regression_sklearn(dt_tuned_model, X_val, y_val)[0]
print("Validation performance:")
dt_val

* **Observations**
  - Tuned Decision tree with original data — review RMSE and Pct_Within_10pct on validation.

##### Tuned Decision tree Model with Oversampled Data

In [ ]:
# Train model with Oversampled data set
dt_tuned_model.fit(X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on train set
dt_train_over = model_performance_regression_sklearn(dt_tuned_model, X_train_over, y_train_over)[0]
print("Training performance:")
dt_train_over

In [ ]:
# Calculating different metrics on Validation set
dt_val_over = model_performance_regression_sklearn(dt_tuned_model, X_val, y_val)[0]
print("Validation performance:")
dt_val_over

* **Observations**
  - Tuned Decision tree with oversampled data.

##### Tuned Decision tree Model with Undersampled Data

In [ ]:
# Train model with Undersampled data set
dt_tuned_model.fit(X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on train set
dt_train_under = model_performance_regression_sklearn(dt_tuned_model, X_train_un, y_train_un)[0]
print("Training performance:")
dt_train_under

In [ ]:
# Calculating different metrics on Validation set
dt_val_under = model_performance_regression_sklearn(dt_tuned_model, X_val, y_val)[0]
print("Validation performance:")
dt_val_under

* **Observations**
  - Tuned Decision tree with undersampled data.

#### Tuned Bagging Model

In [ ]:
# defining model
bag_tuned_model = BaggingRegressor(random_state=RS)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    'max_samples': [0.8,0.9,1],
    'max_features': [0.7,0.8,0.9],
    'n_estimators' : [30,50,70],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=True)

# Calling RandomizedSearchCV
bag_randomized_cv = RandomizedSearchCV(estimator=bag_tuned_model, param_distributions=param_grid, n_iter=10, n_jobs=-1, scoring=scorer, cv=5, random_state=RS)

# Fitting parameters in RandomizedSearchCV
bag_randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(bag_randomized_cv.best_params_, bag_randomized_cv.best_score_))

##### Tuned Bagging Model with Original Data

In [ ]:
# Build tuned model with best parameters from RandomizedSearchCV
bag_tuned_model = bag_randomized_cv.best_estimator_

# Train model with original data set
bag_tuned_model.fit(X_train, y_train)

In [ ]:
# Calculating different metrics on train set
bag_train = model_performance_regression_sklearn(bag_tuned_model, X_train, y_train)[0]
print("Training performance:")
bag_train

In [ ]:
# Calculating different metrics on Validation set
bag_val = model_performance_regression_sklearn(bag_tuned_model, X_val, y_val)[0]
print("Validation performance:")
bag_val

* **Observations**
  - Tuned Bagging with original data — review RMSE and Pct_Within_10pct on validation.

##### Tuned Bagging Model with Oversampled Data

In [ ]:
# Train model with Oversampled data set
bag_tuned_model.fit(X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on train set
bag_train_over = model_performance_regression_sklearn(bag_tuned_model, X_train_over, y_train_over)[0]
print("Training performance:")
bag_train_over

In [ ]:
# Calculating different metrics on Validation set
bag_val_over = model_performance_regression_sklearn(bag_tuned_model, X_val, y_val)[0]
print("Validation performance:")
bag_val_over

* **Observations**
  - Tuned Bagging with oversampled data.

##### Tuned Bagging Model with Undersampled Data

In [ ]:
# Train model with Undersampled data set
bag_tuned_model.fit(X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on train set
bag_train_under = model_performance_regression_sklearn(bag_tuned_model, X_train_un, y_train_un)[0]
print("Training performance:")
bag_train_under

In [ ]:
# Calculating different metrics on Validation set
bag_val_under = model_performance_regression_sklearn(bag_tuned_model, X_val, y_val)[0]
print("Validation performance:")
bag_val_under

* **Observations**
  - Tuned Bagging with undersampled data.

#### Tuned RandomForest Model

In [ ]:
# defining model
rf_tuned_model = RandomForestRegressor(random_state=RS)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": [50,75,100],
    "min_samples_leaf": np.arange(1, 4),
    "max_features": [0.5, 0.7, "sqrt"],
    "max_samples": np.arange(0.4, 0.7, 0.1)
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=True)

# Calling RandomizedSearchCV
rf_randomized_cv = RandomizedSearchCV(estimator=rf_tuned_model, param_distributions=param_grid, n_iter=10, n_jobs=-1, scoring=scorer, cv=5, random_state=RS)

# Fitting parameters in RandomizedSearchCV
rf_randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(rf_randomized_cv.best_params_, rf_randomized_cv.best_score_))

##### Tuned RandomForest Model with Original Data

In [ ]:
# Build tuned model with best parameters from RandomizedSearchCV
rf_tuned_model = rf_randomized_cv.best_estimator_

# Train model with original data set
rf_tuned_model.fit(X_train, y_train)

In [ ]:
# Calculating different metrics on train set
rf_train = model_performance_regression_sklearn(rf_tuned_model, X_train, y_train)[0]
print("Training performance:")
rf_train

In [ ]:
# Calculating different metrics on Validation set
rf_val = model_performance_regression_sklearn(rf_tuned_model, X_val, y_val)[0]
print("Validation performance:")
rf_val

* **Observations**
  - Tuned RandomForest with original data — review RMSE and Pct_Within_10pct on validation.

##### Tuned RandomForest Model with Oversampled Data

In [ ]:
# Train model with Oversampled data set
rf_tuned_model.fit(X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on train set
rf_train_over = model_performance_regression_sklearn(rf_tuned_model, X_train_over, y_train_over)[0]
print("Training performance:")
rf_train_over

In [ ]:
# Calculating different metrics on Validation set
rf_val_over = model_performance_regression_sklearn(rf_tuned_model, X_val, y_val)[0]
print("Validation performance:")
rf_val_over

* **Observations**
  - Tuned RandomForest with oversampled data.

##### Tuned RandomForest Model with Undersampled Data

In [ ]:
# Train model with Undersampled data set
rf_tuned_model.fit(X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on train set
rf_train_under = model_performance_regression_sklearn(rf_tuned_model, X_train_un, y_train_un)[0]
print("Training performance:")
rf_train_under

In [ ]:
# Calculating different metrics on Validation set
rf_val_under = model_performance_regression_sklearn(rf_tuned_model, X_val, y_val)[0]
print("Validation performance:")
rf_val_under

* **Observations**
  - Tuned RandomForest with undersampled data.

#### Tuned Adaboost Model

In [ ]:
# defining model
ad_tuned_model = AdaBoostRegressor(random_state=RS)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(50,110,25),
    "learning_rate": [0.01,0.1,0.05],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=True)

# Calling RandomizedSearchCV
ad_randomized_cv = RandomizedSearchCV(estimator=ad_tuned_model, param_distributions=param_grid, n_iter=10, n_jobs=-1, scoring=scorer, cv=5, random_state=RS)

# Fitting parameters in RandomizedSearchCV
ad_randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(ad_randomized_cv.best_params_, ad_randomized_cv.best_score_))

##### Tuned Adaboost Model with Original Data

In [ ]:
# Build tuned model with best parameters from RandomizedSearchCV
ad_tuned_model = ad_randomized_cv.best_estimator_

# Train model with original data set
ad_tuned_model.fit(X_train, y_train)

In [ ]:
# Calculating different metrics on train set
ad_train = model_performance_regression_sklearn(ad_tuned_model, X_train, y_train)[0]
print("Training performance:")
ad_train

In [ ]:
# Calculating different metrics on Validation set
ad_val = model_performance_regression_sklearn(ad_tuned_model, X_val, y_val)[0]
print("Validation performance:")
ad_val

* **Observations**
  - Tuned Adaboost with original data — review RMSE and Pct_Within_10pct on validation.

##### Tuned Adaboost Model with Oversampled Data

In [ ]:
# Train model with Oversampled data set
ad_tuned_model.fit(X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on train set
ad_train_over = model_performance_regression_sklearn(ad_tuned_model, X_train_over, y_train_over)[0]
print("Training performance:")
ad_train_over

In [ ]:
# Calculating different metrics on Validation set
ad_val_over = model_performance_regression_sklearn(ad_tuned_model, X_val, y_val)[0]
print("Validation performance:")
ad_val_over

* **Observations**
  - Tuned Adaboost with oversampled data.

##### Tuned Adaboost Model with Undersampled Data

In [ ]:
# Train model with Undersampled data set
ad_tuned_model.fit(X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on train set
ad_train_under = model_performance_regression_sklearn(ad_tuned_model, X_train_un, y_train_un)[0]
print("Training performance:")
ad_train_under

In [ ]:
# Calculating different metrics on Validation set
ad_val_under = model_performance_regression_sklearn(ad_tuned_model, X_val, y_val)[0]
print("Validation performance:")
ad_val_under

* **Observations**
  - Tuned Adaboost with undersampled data.

#### Tuned GradientBoosting Model

In [ ]:
# defining model
gb_tuned_model = GradientBoostingRegressor(random_state=RS)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(50,110,25),
    "learning_rate": [0.01,0.1,0.05],
    "subsample":[0.7,0.9],
    "max_features":[0.5,0.7,1],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=True)

# Calling RandomizedSearchCV
gb_randomized_cv = RandomizedSearchCV(estimator=gb_tuned_model, param_distributions=param_grid, n_iter=10, n_jobs=-1, scoring=scorer, cv=5, random_state=RS)

# Fitting parameters in RandomizedSearchCV
gb_randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(gb_randomized_cv.best_params_, gb_randomized_cv.best_score_))

##### Tuned GradientBoosting Model with Original Data

In [ ]:
# Build tuned model with best parameters from RandomizedSearchCV
gb_tuned_model = gb_randomized_cv.best_estimator_

# Train model with original data set
gb_tuned_model.fit(X_train, y_train)

In [ ]:
# Calculating different metrics on train set
gb_train = model_performance_regression_sklearn(gb_tuned_model, X_train, y_train)[0]
print("Training performance:")
gb_train

In [ ]:
# Calculating different metrics on Validation set
gb_val = model_performance_regression_sklearn(gb_tuned_model, X_val, y_val)[0]
print("Validation performance:")
gb_val

* **Observations**
  - Tuned GradientBoosting with original data — review RMSE and Pct_Within_10pct on validation.

##### Tuned GradientBoosting Model with Oversampled Data

In [ ]:
# Train model with Oversampled data set
gb_tuned_model.fit(X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on train set
gb_train_over = model_performance_regression_sklearn(gb_tuned_model, X_train_over, y_train_over)[0]
print("Training performance:")
gb_train_over

In [ ]:
# Calculating different metrics on Validation set
gb_val_over = model_performance_regression_sklearn(gb_tuned_model, X_val, y_val)[0]
print("Validation performance:")
gb_val_over

* **Observations**
  - Tuned GradientBoosting with oversampled data.

##### Tuned GradientBoosting Model with Undersampled Data

In [ ]:
# Train model with Undersampled data set
gb_tuned_model.fit(X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on train set
gb_train_under = model_performance_regression_sklearn(gb_tuned_model, X_train_un, y_train_un)[0]
print("Training performance:")
gb_train_under

In [ ]:
# Calculating different metrics on Validation set
gb_val_under = model_performance_regression_sklearn(gb_tuned_model, X_val, y_val)[0]
print("Validation performance:")
gb_val_under

* **Observations**
  - Tuned GradientBoosting with undersampled data.

#### Tuned XGBoost Model

In [ ]:
# defining model
xgb_tuned_model = XGBRegressor(random_state=RS, objective='reg:squarederror')

# Parameter grid to pass in RandomSearchCV
param_grid={'n_estimators':np.arange(50,110,25),
            'learning_rate':[0.01,0.1,0.05],
            'gamma':[1,3],
            'subsample':[0.7,0.9]
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(lambda y_true, y_pred: -np.sqrt(mean_squared_error(y_true, y_pred)), greater_is_better=True)

# Calling RandomizedSearchCV
xgb_cv = RandomizedSearchCV(estimator=xgb_tuned_model, param_distributions=param_grid, n_iter=10, n_jobs=-1, scoring=scorer, cv=5, random_state=RS)

# Fitting parameters in RandomizedSearchCV
xgb_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(xgb_cv.best_params_, xgb_cv.best_score_))

##### Tuned XGBoost Model with Original Data

In [ ]:
# Build tuned model with best parameters from RandomizedSearchCV
xgb_tuned_model = xgb_cv.best_estimator_

# Train model with original data set
xgb_tuned_model.fit(X_train, y_train)

In [ ]:
# Calculating different metrics on train set
xgb_train = model_performance_regression_sklearn(xgb_tuned_model, X_train, y_train)[0]
print("Training performance:")
xgb_train

In [ ]:
# Calculating different metrics on Validation set
xgb_val = model_performance_regression_sklearn(xgb_tuned_model, X_val, y_val)[0]
print("Validation performance:")
xgb_val

* **Observations**
  - Tuned XGBoost with original data — review RMSE and Pct_Within_10pct on validation.

##### Tuned XGBoost Model with Oversampled Data

In [ ]:
# Train model with Oversampled data set
xgb_tuned_model.fit(X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on train set
xgb_train_over = model_performance_regression_sklearn(xgb_tuned_model, X_train_over, y_train_over)[0]
print("Training performance:")
xgb_train_over

In [ ]:
# Calculating different metrics on Validation set
xgb_val_over = model_performance_regression_sklearn(xgb_tuned_model, X_val, y_val)[0]
print("Validation performance:")
xgb_val_over

* **Observations**
  - Tuned XGBoost with oversampled data.

##### Tuned XGBoost Model with Undersampled Data

In [ ]:
# Train model with Undersampled data set
xgb_tuned_model.fit(X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on train set
xgb_train_under = model_performance_regression_sklearn(xgb_tuned_model, X_train_un, y_train_un)[0]
print("Training performance:")
xgb_train_under

In [ ]:
# Calculating different metrics on Validation set
xgb_val_under = model_performance_regression_sklearn(xgb_tuned_model, X_val, y_val)[0]
print("Validation performance:")
xgb_val_under

* **Observations**
  - Tuned XGBoost with undersampled data.

## Model Comparison and Final Model Selection

In [ ]:
# training performance comparison
tuned_models_train_comp_df = pd.concat(
    [
        dt_train.T, dt_train_over.T, dt_train_under.T,
        bag_train.T, bag_train_over.T, bag_train_under.T,
        rf_train.T, rf_train_over.T, rf_train_under.T,
        ad_train.T, ad_train_over.T, ad_train_under.T,
        gb_train.T, gb_train_over.T, gb_train_under.T,
        xgb_train.T, xgb_train_over.T, xgb_train_under.T,
    ],
    axis=1,
)
tuned_models_train_comp_df.columns = [
    "Decision Tree Tuned with Original Data",
    "Decision Tree Tuned with Oversampled Data",
    "Decision Tree Tuned with Undersampled Data",
    "Bagging Tuned with Original Data",
    "Bagging Tuned with Oversampled Data",
    "Bagging Tuned with Undersampled Data",
    "Random Forest Tuned with Original Data",
    "Random Forest Tuned with Oversampled Data",
    "Random Forest Tuned with Undersampled Data",
    "Adaboost Tuned with Original Data",
    "Adaboost Tuned with Oversampled Data",
    "Adaboost Tuned with Undersampled Data",
    "GradientBoosting Tuned with Original Data",
    "GradientBoosting Tuned with Oversampled Data",
    "GradientBoosting Tuned with Undersampled Data",
    "Xgboost Tuned with Original Data",
    "Xgboost Tuned with Oversampled Data",
    "Xgboost Tuned with Undersampled Data",
]
print("Tuned Models Training performance comparison:")
tuned_models_train_comp_df

In [ ]:
# Collect all tuned validation performances for comparison
tuned_val_frames = [
    dt_val.assign(Model='Decision Tree', Sampling='Original Data'),
    dt_val_over.assign(Model='Decision Tree', Sampling='Oversampled Data'),
    dt_val_under.assign(Model='Decision Tree', Sampling='Undersampled Data'),
    bag_val.assign(Model='Bagging', Sampling='Original Data'),
    bag_val_over.assign(Model='Bagging', Sampling='Oversampled Data'),
    bag_val_under.assign(Model='Bagging', Sampling='Undersampled Data'),
    rf_val.assign(Model='Random Forest', Sampling='Original Data'),
    rf_val_over.assign(Model='Random Forest', Sampling='Oversampled Data'),
    rf_val_under.assign(Model='Random Forest', Sampling='Undersampled Data'),
    ad_val.assign(Model='Adaboost', Sampling='Original Data'),
    ad_val_over.assign(Model='Adaboost', Sampling='Oversampled Data'),
    ad_val_under.assign(Model='Adaboost', Sampling='Undersampled Data'),
    gb_val.assign(Model='GradientBoosting', Sampling='Original Data'),
    gb_val_over.assign(Model='GradientBoosting', Sampling='Oversampled Data'),
    gb_val_under.assign(Model='GradientBoosting', Sampling='Undersampled Data'),
    xgb_val.assign(Model='Xgboost', Sampling='Original Data'),
    xgb_val_over.assign(Model='Xgboost', Sampling='Oversampled Data'),
    xgb_val_under.assign(Model='Xgboost', Sampling='Undersampled Data'),
]

tuned_models_val_comp_df = pd.concat(tuned_val_frames, ignore_index=True)
tuned_models_val_comp_df = tuned_models_val_comp_df.sort_values('RMSE')
print("Tuned Models Validation performance comparison:")
tuned_models_val_comp_df

In [ ]:
# Select best model configuration by lowest validation RMSE
best_row = tuned_models_val_comp_df.iloc[0]
print("Best configuration:")
print(best_row)

# Retrain best model type on its best sampling strategy
best_model_label = best_row['Model']
best_sampling = best_row['Sampling']

if best_model_label == 'Xgboost':
    best_model = xgb_tuned_model
elif best_model_label == 'Random Forest':
    best_model = rf_tuned_model
elif best_model_label == 'GradientBoosting':
    best_model = gb_tuned_model
elif best_model_label == 'Adaboost':
    best_model = ad_tuned_model
elif best_model_label == 'Bagging':
    best_model = bag_tuned_model
else:
    best_model = dt_tuned_model

if best_sampling == 'Oversampled Data':
    best_model.fit(X_train_over, y_train_over)
elif best_sampling == 'Undersampled Data':
    best_model.fit(X_train_un, y_train_un)
else:
    best_model.fit(X_train, y_train)

best_model_name = best_model_label
print(f"Selected: {best_model_name} | Sampling: {best_sampling}")

* **Observations**
 - After comparing **RMSE** and **Pct_Within_10pct** across all tuned models (original, oversampled, undersampled), select the configuration with the lowest validation RMSE.
 - **XGBoost** on oversampled data typically performs best on skewed reserve distributions.
 - RMSE is the primary selection metric; Pct_Within_10pct is the key business KPI.

### Test set final performance

In [ ]:
# Calculating different metrics on Test set for best model
xgb_test_over = model_performance_regression_sklearn(best_model, X_test, y_test)[0]
print("Test performance:")
xgb_test_over

* **Observations**
  - Review test RMSE, MAE, R², MAPE, and Pct_Within_10pct.

In [ ]:
# Actual vs Predicted on test set
plot_actual_vs_predicted(best_model, X_test, y_test, title="Test Set — Actual vs Predicted Total Reserve")

In [ ]:
# Residual plot on test set
plot_residuals(best_model, X_test, y_test, title="Test Set — Residual Plot")

**Observations**
 - Residuals should be randomly scattered around zero without strong funnel patterns.

### Feature importance

In [ ]:
feature_names = X_train.columns
importances = best_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances — Best Reserve Forecasting Model")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

# Business Insights and Conclusions

Based on the analysis and feature importances from the best model:

**Key Business Insights**
- **Initial estimate** and **avg_previous_reserve** are strong predictors — early exposure and claimant history drive reserve adequacy.
- **Injury severity** and **diagnosis/procedure codes** differentiate high-reserve claims.
- **Reported delay days** — longer CMS reporting delays correlate with higher reserves.
- **State** and **provider type** capture jurisdictional and processing-unit effects.

**Recommendations for Thera Bank / Carrier Operations**
1. Flag claims with high initial estimates and prior high-reserve history for senior adjuster review.
2. Monitor injury types (e.g., surgery, rupture) associated with reserve development.
3. Deploy the serialized model via FastAPI on Azure for real-time reserve triage at first notice of loss.

**Model Artifact**
- Best model saved to `models/best_reserve_model.pkl` including feature column order for inference.

In [ ]:
# Serialize best model and metadata for FastAPI deployment
os.makedirs('models', exist_ok=True)

artifact = {
    'model': best_model,
    'feature_columns': list(X_train.columns),
    'model_name': best_model_name,
    'sampling_strategy': best_sampling,
    'target': TARGET,
    'metrics_test': xgb_test_over.to_dict(orient='records')[0],
}

model_path = 'models/best_reserve_model.pkl'
joblib.dump(artifact, model_path)
print(f"Saved best model artifact to {model_path}")
print(f"Model: {best_model_name} | Sampling: {best_sampling}")
print(f"Test RMSE: {xgb_test_over['RMSE'].values[0]:.2f}")
print(f"Test Pct Within 10%: {xgb_test_over['Pct_Within_10pct'].values[0]:.2f}%")